# *QOT Estimator Basic Usage Example*

> This notebook demonstrates the basic usage of the QOT estimator package for optical link.

In [2]:
# Import Dependencies
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import json
from scipy.io import loadmat

# Ignore warnings to have clean cell outputs
import warnings
warnings.filterwarnings("ignore")

### *Create results directory if it doesn't exist*

In [3]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "topology"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\topology


In [5]:
sys.path.append(os.path.abspath(base_dir.parent / 'src'))
from CFM.core.network import Link, LinkParameters, Topology
from CFM.core.band import Band, OpticalParameters
from CFM.core.qot_estimator import ParameterBuilder, ISRSSolver, NLISolver, ASESolver, OSNRCalculator
from CFM.core.qot_optmizer import BasePowerModel, PowerOptimizer, FLPPowerModel, FRPPowerModel, TopologyPowerOptimizer
from CFM.utils.connection_profile import build_connection_profile
from CFM.core.post_process import ModulationConnectionPlotter

In [6]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "topology"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\topology


### *Loading Topology*

In [7]:
from scipy.io import loadmat

data = loadmat(".././data/JPN4812_netCostMatrix.mat")

netcost_matrix = data["netCostMatrix"]


In [8]:
link_params = LinkParameters()

topology = Topology(
    netcost_matrix=netcost_matrix,
    link_params=link_params,
    span_length=80
)


In [9]:
links = topology.get_links_list()

len(links)


82

### *Create spectrum*

In [10]:
my_band_l_params =  OpticalParameters(Rs_mat = 52e9)
my_band_l = Band(
    name='l',
    start_freq = (1.845393470967740e+2 - 0.075/2), # THz
    end_freq = (1.904643470967740e+2 + 0.075/2), # THz
    opt_params = my_band_l_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_l = my_band_l.calc_spectrum() + 0.075*0.5

my_band_c_params =  OpticalParameters(Rs_mat = 52e9)
my_band_c = Band(
    name='c',
    start_freq = (1.909143470967740e+2 - 0.075/2), # THz
    end_freq = (1.968393470967740e+2 + 0.075/2), # THz
    opt_params = my_band_c_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_c = my_band_c.calc_spectrum() + 0.075*0.5

my_band_s_part1_params =  OpticalParameters(Rs_mat = 52e9)
my_band_s_part1 = Band(
    name='s',
    start_freq = (1.972893470967740e+2 - 0.075/2), # THz
    end_freq = (2.035143470967740e+2 + 0.075/2), # THz
    opt_params = my_band_s_part1_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_s_part1 = my_band_s_part1.calc_spectrum() + 0.075*0.5


my_band_s_part2_params =  OpticalParameters(Rs_mat = 52e9)
my_band_s_part2 = Band(
    name='s',
    start_freq = (2.035893870967740e+2 - 0.075/2), # THz
    end_freq = (2.053143870967740e+2 + 0.075/2), # THz
    opt_params = my_band_s_part2_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_s_part2 = my_band_s_part2.calc_spectrum() + 0.075*0.5

grid_center = np.concatenate((spectrum_my_s_part2 , spectrum_my_s_part1, spectrum_my_c , spectrum_my_l))
grid_center = grid_center[::-1]*10e11
bands = [my_band_l, my_band_c, my_band_s_part1, my_band_s_part2]

### *QOT Power Optimizer*

In [11]:
alpha_dB_LCS_268channels = loadmat('.././data/alpha_dB_LCS_268channels.mat')
alpha_dB_LCS = alpha_dB_LCS_268channels['alpha_dB_LCS']

#### *FRP Example*

In [12]:
model = FRPPowerModel()

In [13]:
P_optimizer = TopologyPowerOptimizer(
  topology=topology,
  bands=bands,
  grid_center=grid_center,
  model=model,
  alpha_dB_LCS=alpha_dB_LCS
)

In [14]:
# File path
file_name = results_dir / "Results_FRP_CLSband_JPN4812.csv"

if file_name.exists():
    print("Results already exist. Loading from file...")
    res_df = pd.read_csv(file_name)
else:
    print("Running optimization...")
    res = P_optimizer.optimize()

    # convert dict of dicts correctly
    res_df = pd.DataFrame.from_dict(res, orient="index")
    res_df = res_df.reset_index(names=["u", "v"])

    # save dataframe to csv file
    res_df.to_csv(file_name, index=False)

    print(f"Results saved to {file_name}")


Results already exist. Loading from file...


In [15]:
res_df['start_node'][20]

np.int64(31)

#### *FLP Example*

In [16]:
# model = FLPPowerModel()

In [17]:
# P_optimizer = TopologyPowerOptimizer(
#   topology=topology,
#   bands=bands,
#   grid_center=grid_center,
#   model=model,
#   alpha_dB_LCS=alpha_dB_LCS
# )

In [18]:
# res = P_optimizer.optimize()

In [19]:
# res

### *Test scenario*

In [20]:
G = topology.get_graph()

In [21]:
_,All_connections_Profile, degree_node_all_topo =build_connection_profile(G=G.copy(), core_nodes=np.array([7,8,9,11,19,23,25,26,32,35,36,40]), kSP=3, Lspan=80)

In [22]:
All_connections_Profile

array([[np.int64(7), np.int64(8),
        array([array([20., 75., 69., 70., 64., 22.]),
               array([18., 42., 69., 70., 64., 22.]),
               array([19., 40., 42., 69., 70., 64., 22.])], dtype=object),
        array([551.3, 590.1, 608. ]), array([ 9., 10., 11.]),
        array([list([np.int64(7), 36, 35, 31, 44, 27, np.int64(8)]),
               list([np.int64(7), 15, 35, 31, 44, 27, np.int64(8)]),
               list([np.int64(7), 16, 15, 35, 31, 44, 27, np.int64(8)])],
              dtype=object)                                              ,
        array([3., 4., 4.])],
       [np.int64(7), np.int64(9),
        array([array([19., 43., 36., 33., 34., 23.]),
               array([19., 43., 36., 38., 39., 23.]),
               array([18., 41., 35., 33., 34., 23.])], dtype=object),
        array([513.6, 526.9, 544.5]), array([10., 10., 11.]),
        array([list([np.int64(7), 16, 30, 13, 12, 14, np.int64(9)]),
               list([np.int64(7), 16, 30, 13, 43, 14, np.int6

#### Defining SNR Params

In [23]:
from scipy.special import erfcinv

# GSNR Calculation
target_ber = 1.3e-2

target_SNR_dB_2 = 10*np.log10(1*(erfcinv(2*target_ber))**2)       # PM-BPSK
target_SNR_dB_4 = 10*np.log10(2*(erfcinv(2*target_ber))**2)       # PM-QPSK
target_SNR_dB_8 = 10*np.log10((14/3)*(erfcinv(1.5*target_ber))**2) # PM-8QAM
target_SNR_dB_16 = 10*np.log10((10)*(erfcinv((8/3)*target_ber))**2) # PM-16QAM
M = 32  # PM-32QAM
target_SNR_dB_32 = 10*np.log10(2*(erfcinv(np.log2(M)*target_ber/2/(1-1/np.sqrt(M))))**2*(M-1)/3)
M = 64  # PM-64QAM
target_SNR_dB_64 = 10*np.log10(2*(erfcinv(np.log2(M)*target_ber/2/(1-1/np.sqrt(M))))**2*(M-1)/3)

target_SNR_dB = [target_SNR_dB_2, target_SNR_dB_4, target_SNR_dB_8, target_SNR_dB_16, target_SNR_dB_32, target_SNR_dB_64]

In [24]:
import numpy as np

def parse_opt(x):
    if isinstance(x, str):
        return eval(x, {"np": np, "array": np.array})
    return x

res_df['optimization'] = res_df['optimization'].apply(parse_opt)


In [25]:
Results_FRP_CLSband_JPN4812 = res_df
GSNR_opt_span_FRP_LCS = np.vstack(res_df['optimization'].apply(lambda d: d['gsnr']))
[num_links, num_channels] = np.shape(GSNR_opt_span_FRP_LCS)
idxs = np.array([d.get('idx') for _, _, d in G.edges(data=True)])
wss_penalty_degree = np.array([0, 0.5, 0.75, 1, 1.25, 1.5, 1.75, 3.5, 4, 5])
kSP=3


In [26]:
GSNR_connection = np.zeros((len(All_connections_Profile[:,0]), num_channels, kSP))
Modulation_connection = np.zeros((len(All_connections_Profile[:,0]), num_channels, kSP))
Results_K3SP_FRP_SLC_CBG_JPN4812 = {
    'All_connections_Profile': All_connections_Profile,
    'Modulation_connection': Modulation_connection,
    'GSNR_connection': GSNR_connection
}
MFC_6=0; MFC_5=0; MFC_4=0; MFC_3=0; MFC_2=0; MFC_1=0; MCF_0=0;

for connection_counter in range(len(All_connections_Profile[:,0])):

    num_amp_shp = All_connections_Profile[connection_counter,6]
    node_list_allpath = All_connections_Profile[connection_counter,5]
    link_list_allpath = All_connections_Profile[connection_counter,2]

    for chanel_counter in range(num_channels):

        for path_counter in range(kSP):
            GSNR_BVT = 0
            node_list = node_list_allpath[path_counter]
            link_list_shp = link_list_allpath[path_counter]

            for link_counter in range(len(link_list_shp)):

                idx = np.where(idxs == int(link_list_shp[link_counter]))[0][0]
                GSNR_Ch = GSNR_opt_span_FRP_LCS[idx, chanel_counter]
                start_node = res_df['start_node'][idx]
                end_node = res_df['end_node'][idx]

                for span_counter in range(G[start_node][end_node]['link'].num_span):

                    GSNR_BVT  = GSNR_BVT + (10**(GSNR_Ch/10))**-1

                GSNR_BVT = GSNR_BVT/10**(wss_penalty_degree[int(degree_node_all_topo[int(node_list[link_counter+1])-1])-1]/10)
            
            GSNR_connection2 = 10*np.log10((GSNR_BVT + 10**-3.6)**-1)
            GSNR_connection3 =  GSNR_connection2 - 1 - 0.05*num_amp_shp[path_counter] - 0*10*np.log10(len(node_list)-1)
            GSNR_connection[connection_counter,chanel_counter,path_counter] = GSNR_connection3

            if (GSNR_connection3 >= target_SNR_dB[5]):
                MFC_6 = MFC_6 + 1
                Modulation_connection[connection_counter, chanel_counter, path_counter] = 6

            elif (target_SNR_dB[4] <= GSNR_connection3 and GSNR_connection3 < target_SNR_dB[5]):
                MFC_5 = MFC_5+1
                Modulation_connection[connection_counter,chanel_counter, path_counter] = 5

            elif (target_SNR_dB[3] <= GSNR_connection3 and GSNR_connection3 < target_SNR_dB[4]):
                MFC_4 = MFC_4+1
                Modulation_connection[connection_counter,chanel_counter, path_counter] = 4

            elif (target_SNR_dB[2] <= GSNR_connection3 and GSNR_connection3 < target_SNR_dB[3]):
                MFC_3 = MFC_3+1
                Modulation_connection[connection_counter,chanel_counter, path_counter] = 3

            elif (target_SNR_dB[1] <= GSNR_connection3 and GSNR_connection3 < target_SNR_dB[2]):
                MFC_2 = MFC_2+1
                Modulation_connection[connection_counter,chanel_counter, path_counter] = 2

            elif (target_SNR_dB[0] <= GSNR_connection3 and GSNR_connection3 < target_SNR_dB[1]):
                MFC_1 = MFC_1+1
                Modulation_connection[connection_counter,chanel_counter, path_counter] = 1

            elif (target_SNR_dB[0] > GSNR_connection3):
                MFC_0 = MFC_0 + 1
                Modulation_connection[connection_counter,chanel_counter, path_counter] = 0

Results_K3SP_FRP_SLC_CBG_JPN4812['All_connections_Profile'] = All_connections_Profile
Results_K3SP_FRP_SLC_CBG_JPN4812['Modulation_connection'] = Modulation_connection
Results_K3SP_FRP_SLC_CBG_JPN4812['GSNR_connection'] = GSNR_connection

In [27]:
path_GSNR = np.zeros((len(Modulation_connection[:,0]), len(Modulation_connection[0,0,:])))
for i in range(len(Modulation_connection[:,0,0])):
    for j in range(len(Modulation_connection[0,0,:])):
        path_GSNR[i,j] = np.mean(GSNR_connection[i,:,j])

ML6_path_1 = len(np.where(Modulation_connection[:,:,0]==6)[0])
ML5_path_1 = len(np.where(Modulation_connection[:,:,0]==5)[0])
ML4_path_1 = len(np.where(Modulation_connection[:,:,0]==4)[0])
ML3_path_1 = len(np.where(Modulation_connection[:,:,0]==3)[0])
ML2_path_1 = len(np.where(Modulation_connection[:,:,0]==2)[0])
ML1_path_1 = len(np.where(Modulation_connection[:,:,0]==1)[0])

ML_per_path = np.zeros((6, kSP))
for i in range(6):
    for j in range(kSP):
        ML_per_path[i,j] = len(np.where(Modulation_connection[:,:,j]==(i+1))[0])

ML_per_path = 100*ML_per_path/(len(Modulation_connection[:,0,0])*len(Modulation_connection[0,:,0]))

for i in range(6):
    for j in range(kSP):
        ML_per_path[i,j] = len(np.where(Modulation_connection[:,:,j]==(i+1))[0])*(i+1)
FRP_JPN = sum(ML_per_path)*1e-3*100

In [28]:
FRP_JPN

array([10259.3, 10209.7, 10176.1])

In [32]:
plotter = ModulationConnectionPlotter(Modulation_connection)
fig = plotter.plot(k_index=0)
fig.show()